# 7 Serving.ipynb — versión ajustada

Corrige el punto opcional del entregable "Trabajar el modelo lambda en las capas":

> sumariza los **últimos 20 minutos** de la capa BATCH (CLEANSED) junto con los **últimos 2 minutos** de la capa SPEED

Diferencia respecto a la versión anterior documentada en el informe: antes se usaba una sola ventana ampliada a 6 horas
(por falta de datos recientes durante el troubleshooting). Aquí se usan **dos ventanas independientes**, cada una acorde
a la cadencia real de su capa de origen (BATCH se actualiza cada 10/20 min, SPEED cada 2 min).

**Importante antes de correr esta celda:** para que la ventana de SPEED de 2 minutos devuelva registros, el notebook
`5 Streaming.ipynb` y el emulador de datos (`emulador_datos.py` / equivalente) deben estar corriendo en background
justo antes de ejecutar este notebook. Si no hay streaming activo, `SPEED raw count` va a dar 0 (mismo problema que
se documentó en la sección 5.1 del informe original).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, expr

spark = (
    SparkSession.builder
    .appName("ServingAjustado")
    .getOrCreate()
)

BATCH_PATH = "hdfs://namenode:8020/lambda/cleansed/batch/ventas"
SPEED_PATH = "hdfs://namenode:8020/lambda/speed"
SERVING_PATH = "hdfs://namenode:8020/lambda/serving"

## 1. Lectura BATCH — ventana de 20 minutos

In [ ]:
ventas_batch = spark.read.parquet(BATCH_PATH)

ventas_batch_reciente = ventas_batch.where(
    "order_date >= current_timestamp() - INTERVAL 20 MINUTES"
)

print("BATCH raw count (sin filtro):", ventas_batch.count())
print("BATCH últimos 20 min:", ventas_batch_reciente.count())
ventas_batch_reciente.select("order_date").orderBy("order_date", ascending=False).show(5, truncate=False)

## 2. Lectura SPEED — ventana de 2 minutos

In [ ]:
ventas_speed = spark.read.parquet(SPEED_PATH)

ventas_speed_reciente = ventas_speed.where(
    "order_date >= current_timestamp() - INTERVAL 2 MINUTES"
)

print("SPEED raw count (sin filtro):", ventas_speed.count())
print("SPEED últimos 2 min:", ventas_speed_reciente.count())

if ventas_speed_reciente.count() == 0:
    print("ADVERTENCIA: 0 registros en la ventana de 2 min de SPEED.")
    print("Verificar que 5 Streaming.ipynb y el emulador de datos estén corriendo ANTES de ejecutar esta celda.")
else:
    ventas_speed_reciente.select("order_date").orderBy("order_date", ascending=False).show(5, truncate=False)

## 3. Unificación BATCH + SPEED

In [ ]:
ventas_unificadas = ventas_batch_reciente.withColumn("fuente", expr("'BATCH'")) \
    .unionByName(
        ventas_speed_reciente.withColumn("fuente", expr("'SPEED'")),
        allowMissingColumns=True
    )

print("Total unificado (BATCH 20min + SPEED 2min):", ventas_unificadas.count())

## 4. Sumarizaciones (por fuente, por estado, por producto)

In [ ]:
resumen_por_fuente = ventas_unificadas.groupBy("fuente") \
    .agg({"order_item_id": "count", "order_item_subtotal": "sum"})

resumen_por_status = ventas_unificadas.groupBy("order_status") \
    .agg({"order_item_id": "count", "order_item_subtotal": "sum"})

resumen_por_producto = ventas_unificadas.groupBy("order_item_product_id") \
    .agg({"order_item_subtotal": "sum"}).orderBy("sum(order_item_subtotal)", ascending=False)

print("--- Resumen por fuente ---")
resumen_por_fuente.show(truncate=False)

print("--- Resumen por estado ---")
resumen_por_status.show(truncate=False)

print("--- Top 10 productos ---")
resumen_por_producto.show(10, truncate=False)

## 5. Guardado en HDFS (SERVING)

In [ ]:
resumen_por_fuente.write.mode("overwrite").parquet(f"{SERVING_PATH}/resumen_por_fuente")
resumen_por_status.write.mode("overwrite").parquet(f"{SERVING_PATH}/resumen_por_status")
resumen_por_producto.write.mode("overwrite").parquet(f"{SERVING_PATH}/resumen_por_producto")

print("Resumen guardado en", SERVING_PATH)
print("Ventanas usadas -> BATCH: 20 minutos | SPEED: 2 minutos")

## 6. Validación en HDFS (correr en la terminal del Codespace, no en esta celda)

```bash
docker exec -it namenode hdfs dfs -ls /lambda/serving
docker exec -it namenode hdfs dfs -ls /lambda/serving/resumen_por_fuente
docker exec -it namenode hdfs dfs -ls /lambda/serving/resumen_por_status
docker exec -it namenode hdfs dfs -ls /lambda/serving/resumen_por_producto
```

Capturar screenshot de la salida de consola (conteos BATCH/SPEED de las celdas 1 y 2) y del `hdfs dfs -ls` de arriba
para reemplazar la sección 5 del informe PDF (que actualmente muestra la versión con ventana de 6 horas).